In [4]:
# transformer-lens 3.4.0 pins torchvision>=0.22,<0.23 (for torch 2.7.x),
# but Colab has torch 2.10 + torchvision 0.25. This breaks pip's resolver.
# Workaround: install both with --no-deps, then install dependencies separately.


# Step 1: Install sae-lens and transformer-lens without the torchvision constraint
!pip install --no-deps transformer-lens sae-lens

# Step 2: Install all actual dependencies (minus torchvision, not needed for text SAEs)
!pip install \
    "transformers>=5.4.0,<6.0.0" \
    "datasets>=3.1.0" \
    "einops>=0.6.0" \
    "jaxtyping>=0.2.11" \
    "fancy-einsum>=0.0.3" \
    "beartype>=0.14.1" \
    "accelerate>=0.23.0" \
    "rich>=12.6.0" \
    "sentencepiece" \
    "typeguard>=4.2,<5" \
    "wandb>=0.13.5" \
    "transformers-stream-generator>=0.0.5,<0.1" \
    "better-abc>=0.0.3" \
    "protobuf>=3.20.0" \
    "huggingface-hub>=0.23.2" \
    "safetensors>=0.4.2,<1.0.0" \
    "plotly>=5.19.0" \
    "nltk>=3.8.1" \
    "python-dotenv>=1.0.1" \
    "pyyaml>=6.0.1" \
    "simple-parsing>=0.1.8" \
    "tenacity>=9.0.0" \
    "babe>=0.0.7"

In [9]:
from __future__ import annotations

import argparse
import re
import sys
from dataclasses import dataclass
from concurrent.futures import ThreadPoolExecutor, as_completed
from typing import Iterable

import requests
import torch
from sae_lens import SAE
from transformer_lens import HookedTransformer

print(torch.cuda.is_available())
print(torch.cuda.current_device())
print(torch.cuda.get_device_name(0))
print(torch.cuda.is_bf16_supported())

True
0
Tesla T4
True


In [10]:
DEFAULT_RELEASE = "gemma-scope-2b-pt-res"
DEFAULT_SAE_ID = "layer_20/width_16k/average_l0_71"
DEFAULT_MODEL = "gemma-2-2b"
NEURONPEDIA_FEATURE_API = "https://www.neuronpedia.org/api/feature"

DEVICE = 'cuda'
DTYPE = torch.bfloat16


In [11]:
print("----loading SAE----")
sae = SAE.from_pretrained(
    release=DEFAULT_RELEASE,
    sae_id=DEFAULT_SAE_ID,
    device=DEVICE,
    dtype=DTYPE
)

----loading SAE----


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:134: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


layer_20/width_16k/average_l0_71/params.(…):   0%|          | 0.00/302M [00:00<?, ?B/s]

In [ ]:
print("----loading hooks----")
metadata = getattr(sae.cfg, "metadata", None)
hook_name = getattr(metadata, "hook_name", None)
match = re.search(r"blocks\.(\d+)\.", hook_name)
hook_layer = int(match.group(1)) if match else None

model_name = getattr(metadata, "model_name", None)
prepend_bos = getattr(metadata, "prepend_bos", None)

print(f"\tHook name:\t{hook_name}")
print(f"\tHook layer:\t{hook_layer}")
print(f"\tModel name:\t{model_name}")
print(f"\tPrepend BOS:\t{prepend_bos}")


----loading hooks----
	Hook name:	blocks.20.hook_resid_post
	Hook layer:	20
	Model name:	gemma-2-2b
	Prepend BOS:	True
